In [1]:
import os
import sys
sys.stdout.flush()
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import SystemMessage, HumanMessage, AIMessageChunk
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

In [2]:
load_dotenv()

True

In [3]:
llm = ChatOpenAI(
    model="nvidia/nemotron-3.5-lightning-30b-a3b",
    api_key=os.getenv("NVIDIA_API_KEY"),
    base_url="https://integrate.api.nvidia.com/v1",
    temperature=0
)

In [4]:
class State(TypedDict):
    message: Annotated[list, add_messages]

SYSTEM_PROMPT = SystemMessage(content="You are a helpful assistant.")    

In [5]:
def chat_node(state: State):
    response = llm.invoke([SYSTEM_PROMPT] + state["message"])
    return {"message": [response]}

In [6]:
graph = StateGraph(State)

graph.add_node("chat_node",chat_node)

graph.add_edge(START,"chat_node")
graph.add_edge("chat_node",END)

workflow = graph.compile(checkpointer=MemorySaver())
config = {"configurable": {"thread_id": "chat-1"}}

In [ ]:

while True:
    try:
        user_input = input("You: ").strip()
    except (KeyboardInterrupt, EOFError):
        print("\nGoodbye!")
        break

    if not user_input:
        continue

    if user_input.lower() in ["exit", "quit"]:
        print("Goodbye!")
        break

    if user_input.lower() == "history":
        saved_messages = workflow.get_state(config).values.get(
            "message", []
        )

        for message in saved_messages:
            print(f"{message.type} -> {message.content}")

        print()
        continue

    # IMPORTANT: This block must be inside the while loop
    try:
        result = workflow.invoke(
            {"message": [HumanMessage(content=user_input)]},
            config
        )

        print("Bot:", result["message"][-1].content, flush=True)

        snapshot = workflow.get_state(config)
        saved_messages = snapshot.values.get("message", [])

        print("\n--- Conversation History ---")
        for message in saved_messages:
            if message.type == "human":
                print(f"You: {message.content}")
            elif message.type == "ai":
                print(f"Bot: {message.content}")

        print("----------------------------\n", flush=True)

    except Exception as error:
        print(
            f"Something went wrong: {type(error).__name__}: {error}",
            flush=True
        )


Bot: Hello! How can I help you today?

--- Conversation History ---
You: hello
Bot: Hello! How can I help you today?
----------------------------

Bot: I'm doing well, thank you for asking! 😊 How are you doing, and how can I help you today?

--- Conversation History ---
You: hello
Bot: Hello! How can I help you today?
You: how are you
Bot: I'm doing well, thank you for asking! 😊 How are you doing, and how can I help you today?
----------------------------

